In [4]:
!git clone https://github.com/ahmedmujtaba39/Wahm.git
%cd Wahm

Cloning into 'Wahm'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 156 (delta 62), reused 109 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 788.19 KiB | 2.42 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/Wahm


In [7]:
%cd /content/Wahm

/content/Wahm


In [8]:
!ls translate.py

translate.py


In [9]:
import csv
for d in ["gulf", "sudanese",]:
    rows = list(csv.DictReader(open(f"exemplars_{d}.csv", encoding="utf-8")))
    filled = sum(1 for r in rows if r.get("dialect_question","").strip())
    print(f"{d}: {filled}/10 exemplars filled")

gulf: 10/10 exemplars filled
sudanese: 10/10 exemplars filled


In [27]:
%cd /content/Wahm
import translate
from openai import AzureOpenAI
import time

translate._client = AzureOpenAI(
    api_key="1vFwfHwfJ8szELpSZvAAyEbrE3T3NyEKlmK7AdnTkFjASmeWAPBJJQQJ99CHACfhMk5XJ3w3AAAAACOGhCHR",
    api_version="2024-12-01-preview",
    azure_endpoint="https://mohammed-claude.services.ai.azure.com/",
)
translate.MODEL = "gpt-5.6-sol"

def patched_call(msgs, temperature, *args, **kwargs):
    retries = 3
    for attempt in range(retries):
        try:
            r = translate._client.chat.completions.create(
                model=translate.MODEL,
                messages=msgs,
                max_completion_tokens=300,
                # temperature removed — Azure deployment only supports default (1)
            )
            content = r.choices[0].message.content.strip()
            model_used = r.model or translate.MODEL
            return content, model_used
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

translate.call = patched_call
print("Patched OK (no temperature)")

/content/Wahm
Patched OK (no temperature)


In [28]:
translate.translate("gulf", limit=20)

Gulf (Khaleeji) Arabic
  exemplars   : 10
  sub-variety : UNSPECIFIED (ask your validator)
  CODA rule   : on
  temperature : 0.3
  model       : gpt-5.6-sol
  10/20 new (0 already complete)
  20/20 new (0 already complete)

wrote candidates_gulf.csv
next: python qc.py gulf


In [29]:
import importlib, qc
importlib.reload(qc)
qc.run("gulf")

gulf: 20 candidates
  mean back-translation similarity : 0.849
  mean dialect distance            : 0.688
  flagged rows: 1/20
    MEANING_DRIFT: 1

wrote validation_gulf.csv — send this to the validator


In [ ]:
translate.translate("gulf")

Gulf (Khaleeji) Arabic
  exemplars   : 10
  sub-variety : UNSPECIFIED (ask your validator)
  CODA rule   : on
  temperature : 0.3
  model       : gpt-5.6-sol
  10/280 new (20 already complete)
  20/280 new (20 already complete)
  30/280 new (20 already complete)
  40/280 new (20 already complete)
  50/280 new (20 already complete)
  60/280 new (20 already complete)
  70/280 new (20 already complete)
  80/280 new (20 already complete)
  90/280 new (20 already complete)
  100/280 new (20 already complete)
  110/280 new (20 already complete)
  120/280 new (20 already complete)
  130/280 new (20 already complete)
  140/280 new (20 already complete)
  150/280 new (20 already complete)
  160/280 new (20 already complete)
  170/280 new (20 already complete)
  180/280 new (20 already complete)
  190/280 new (20 already complete)
  200/280 new (20 already complete)
  210/280 new (20 already complete)
  220/280 new (20 already complete)
  230/280 new (20 already complete)
  240/280 new (20 alrea

In [12]:
# STEP 1: Test Azure connection
!pip install -q openai

from openai import AzureOpenAI

# Fill in YOUR values
API_KEY = "1vFwfHwfJ8szELpSZvAAyEbrE3T3NyEKlmK7AdnTkFjASmeWAPBJJQQJ99CHACfhMk5XJ3w3AAAAACOGhCHR"
ENDPOINT = "https://mohammed-claude.services.ai.azure.com/"
DEPLOYMENT = "gpt-5.6-sol"   # try this first, change if it errors

client = AzureOpenAI(
    api_key=API_KEY,
    api_version="2024-12-01-preview",
    azure_endpoint=ENDPOINT,
)

# Quick test
try:
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": "قل مرحبا بالعربي"}],
        max_tokens=50,
    )
    print("SUCCESS:", r.choices[0].message.content)
    print("Model works:", DEPLOYMENT)
except Exception as e:
    print("FAILED:", e)
    print("\nIf 'deployment not found', try changing DEPLOYMENT to one of:")
    print("  gpt-4o-mini, gpt-4, gpt-35-turbo, claude-3-5-sonnet")

FAILED: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}

If 'deployment not found', try changing DEPLOYMENT to one of:
  gpt-4o-mini, gpt-4, gpt-35-turbo, claude-3-5-sonnet


In [14]:
from openai import AzureOpenAI

API_KEY = "1vFwfHwfJ8szELpSZvAAyEbrE3T3NyEKlmK7AdnTkFjASmeWAPBJJQQJ99CHACfhMk5XJ3w3AAAAACOGhCHR"
ENDPOINT = "https://mohammed-claude.services.ai.azure.com/"
DEPLOYMENT = "gpt-5.6-sol"

client = AzureOpenAI(
    api_key=API_KEY,
    api_version="2024-12-01-preview",
    azure_endpoint=ENDPOINT,
)

r = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{"role": "user", "content": "قل مرحبا بالعربي"}],
    max_completion_tokens=50,
)
print("SUCCESS:", r.choices[0].message.content)

SUCCESS: مرحبًا!
